In [1]:
# ============================================
# NEWQ: End-to-End Pipeline in Google Colab
# OCR -> Summarize -> MCQ Generation -> Classifier Training & Inference
# ============================================

# --- 0) Setup & Installs ---
!pip -q install easyocr==1.7.1 transformers==4.44.2 datasets==2.21.0 sentencepiece==0.2.0 \
                 accelerate==0.34.2 scikit-learn==1.5.1 openai==1.46.0 \
                 torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip -q install spacy==3.7.2 uvicorn==0.30.6 fastapi==0.114.0
!python -m spacy download en_core_web_sm

import os, json, random, textwrap, re
from typing import List, Dict, Tuple

import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

import easyocr
import numpy as np
import pandas as pd
import spacy

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding,
    pipeline, set_seed
)
from datasets import Dataset

set_seed(42)

# Optional: Set your OpenAI key for LLM MCQ generation (otherwise offline method will be used)
os.environ["OPENAI_API_KEY"] = ""  # <-- paste key or leave empty to stay offline

# --- Colab convenience (upload images) ---
from google.colab import files

ERROR: Could not find a version that satisfies the requirement easyocr==1.7.1 (from versions: none)
ERROR: No matching distribution found for easyocr==1.7.1


^C


c:\Users\hp\AppData\Local\Programs\Python\Python313\python.exe: No module named spacy


ModuleNotFoundError: No module named 'torch'

  error: subprocess-exited-with-error
  
  × installing build dependencies for spacy did not run successfully.
  │ exit code: 1
  ╰─> [103 lines of output]
      Ignoring numpy: markers 'python_version < "3.9"' don't match your environment
        Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
        Using cached Cython-0.29.37-py2.py3-none-any.whl.metadata (3.1 kB)
        Using cached cymem-2.0.13-cp313-cp313t-win_amd64.whl.metadata (9.9 kB)
        Using cached preshed-3.0.12.tar.gz (15 kB)
        Installing build dependencies: started
        Installing build dependencies: finished with status 'done'
        Getting requirements to build wheel: started
        Getting requirements to build wheel: finished with status 'done'
        Preparing metadata (pyproject.toml): started
        Preparing metadata (pyproject.toml): finished with status 'done'
        Using cached murmurhash-1.0.15-cp313-cp313t-win_amd64.whl.metadata (2.3 kB)
        Using cached thinc-8.2.